# AWP Full Showcase -- Every Feature in One Notebook

This capstone notebook demonstrates **every feature** of the Agent Workflow Protocol (AWP) in a single, integrated walkthrough. After completing this notebook you will understand:

- All 7 semantic layers and how they compose into a workflow
- How to build a complete workflow programmatically from scratch
- Schema validation, graph validation, rule checking, and compliance auditing
- Running data-driven workflows with `AgentWorkflow`
- All runtime components wired together (tools, code execution, messaging, security, observability, state)
- The full autonomy spectrum (A0 through A4) with side-by-side comparison

**Prerequisites**: Notebooks 01-05 cover individual topics in depth. This notebook ties them all together.

---

## 1. Provider Setup

Configure your LLM provider. Supported options:
- **Ollama** -- local, no API key needed
- **OpenRouter** -- cloud, requires `OPENROUTER_API_KEY`
- **Custom API** -- any OpenAI-compatible endpoint

In [1]:
# ============================================================
# Provider Selection -- choose ONE of: "ollama", "openrouter", "custom"
# ============================================================
PROVIDER = "openrouter"  # <-- change this

# --- Ollama (local) -------------------------------------------
OLLAMA_MODEL = "qwen3:1.7b"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# --- OpenRouter (cloud) ---------------------------------------
OPENROUTER_API_KEY = ""  # paste your key or set env var OPENROUTER_API_KEY
OPENROUTER_MODEL = "openai/gpt-5-mini"

# --- Custom OpenAI-compatible API -----------------------------
CUSTOM_API_KEY = ""
CUSTOM_BASE_URL = ""
CUSTOM_MODEL = ""

# ==============================================================
# DO NOT EDIT BELOW -- wires up the selected provider
# ==============================================================
import os

if PROVIDER == "ollama":
    os.environ["LLM_API_KEY"] = "ollama"
    os.environ["LLM_BASE_URL"] = OLLAMA_BASE_URL
    MODEL = f"ollama/{OLLAMA_MODEL}"
    print(f"Using Ollama  model={OLLAMA_MODEL}  url={OLLAMA_BASE_URL}")

elif PROVIDER == "openrouter":
    key = OPENROUTER_API_KEY or os.getenv("OPENROUTER_API_KEY", "")
    if not key:
        raise ValueError("Set OPENROUTER_API_KEY above or as an environment variable")
    os.environ["LLM_API_KEY"] = key
    os.environ["LLM_BASE_URL"] = "https://openrouter.ai/api/v1"
    MODEL = OPENROUTER_MODEL
    print(f"Using OpenRouter  model={OPENROUTER_MODEL}")

elif PROVIDER == "custom":
    key = CUSTOM_API_KEY or os.getenv("LLM_API_KEY", "")
    url = CUSTOM_BASE_URL or os.getenv("LLM_BASE_URL", "")
    if not key:
        raise ValueError("Set CUSTOM_API_KEY above or LLM_API_KEY as an environment variable")
    if not url:
        raise ValueError("Set CUSTOM_BASE_URL above or LLM_BASE_URL as an environment variable")
    os.environ["LLM_API_KEY"] = key
    os.environ["LLM_BASE_URL"] = url
    MODEL = CUSTOM_MODEL
    print(f"Using custom API  model={CUSTOM_MODEL}  url={url}")

else:
    raise ValueError(f"Unknown PROVIDER '{PROVIDER}'. Use 'ollama', 'openrouter', or 'custom'.")

Using OpenRouter  model=openrouter/google/gemini-2.5-flash


In [2]:
# Common imports and project root setup
import json
import tempfile
from pathlib import Path

PROJECT_ROOT = Path("/home/shumway/projects/agent-workflow-protocol")
os.chdir(PROJECT_ROOT)

import awp
print(f"AWP version: {awp.__version__}")
print(f"Working directory: {os.getcwd()}")
print(f"Project root: {PROJECT_ROOT}")

AWP version: 1.0.0
Working directory: /home/shumway/projects/agent-workflow-protocol
Project root: /home/shumway/projects/agent-workflow-protocol


---

## 2. The AWP Feature Map

AWP organizes everything into **7 semantic layers** that span an **autonomy spectrum** from A0 (fully prescribed) to A4 (self-organizing). Here is the complete feature map:

### The 7 Layers

| Layer | Name | Key Features | Config Model |
|-------|------|--------------|-------------|
| 0 | **Manifest** | Workflow name, version, description, tags, author, dependencies, env vars, settings | `AWPManifest`, `WorkflowMetadata` |
| 1 | **Identity** | Agent ID, role, description, model config, prompt architecture, output contract | `AWPAgent` |
| 2 | **Capabilities** | Tools (MCP), skills, code execution, data sources, dynamic tool creation | `ToolsCapability`, `CustomToolsConfig` |
| 3 | **Communication** | Message bus (internal/Redis/NATS), channels (direct/broadcast/topic), patterns (pub-sub/request-response) | `CommunicationConfig`, `Channel` |
| 4 | **Memory** | Long-term (MEMORY.md), daily logs, episodic, semantic (vector DB), curation | `MemoryConfig` |
| 5 | **Orchestration** | DAG engine, delegation loop engine, state sharing, budgets, worker policies, validation gates | `AWPOrchestrationConfig`, `DelegationLoopConfig` |
| 6 | **Observability** | Logging, metrics (counters/gauges/histograms), distributed tracing, audit trail (hash-chained), health checks | `ObservabilityConfig` |

### Cross-Cutting: Security

| Feature | Description |
|---------|------------|
| Circuit Breaker | Opens after N failures, resets after timeout, half-open testing |
| Rate Limiter | Sliding window, per-agent or global, calls-per-minute |
| Access Controller | Per-agent tool deny lists, default allow/deny policy |
| Secrets | Environment, file, vault, AWS Secrets Manager backends |

### The Autonomy Spectrum

| Level | Name | Description | Key Requirement |
|-------|------|-------------|----------------|
| A0 | Prescribed | Static DAG, fixed agents, fixed order | Manifest + 1 agent + output contract |
| A1 | Adaptive | Conditional execution, loops, fan-out, multi-agent DAG | Multi-agent graph OR conditional features |
| A2 | Delegating | Manager spawns workers dynamically | Delegation loop engine + budget |
| A3 | Self-Tooling | Agents create tools and skills at runtime | `dynamic_tools.enabled: true` + safety envelope |
| A4 | Self-Organizing | Recursive delegation with budget distribution | `max_depth > 1` + observability |

### Runtime Components

| Component | Class | Purpose |
|-----------|-------|---------|
| Tool Registry | `ToolRegistry` | Built-in + custom MCP tools, dynamic tool creation |
| Code Executor | `CodeExecutor` | Subprocess-based Python sandbox |
| Docker Executor | `DockerExecutor` | Container-isolated code execution |
| Venv Executor | `VenvExecutor` | Virtual environment executor |
| Message Bus | `MessageBus` | Inter-agent direct/broadcast/channel messaging |
| Security Context | `SecurityContext` | Circuit breaker + rate limiter + access control |
| Observability Context | `ObservabilityContext` | Tracer + metrics + audit trail |
| State Persistence | `StatePersistence` | JSON-based per-agent checkpoints + final snapshot |
| Workflow Runner | `WorkflowRunner` | DAG topological execution engine |
| Delegation Loop Runner | `DelegationLoopRunner` | Manager-worker loop engine |

In [3]:
# Verify all key imports in one place -- this proves everything is available
from awp.parser import parse_manifest, parse_agent, resolve_templates
from awp.validator import (
    validate_schema, validate_graph, validate_contracts,
    check_compliance, validate_rules, AutonomyLevel,
)
from awp.models import (
    AWPManifest, AWPAgent, AWPOrchestrationConfig, GraphNode,
    DelegationLoopConfig, DelegationBudget, StateModel, SharingConfig,
    MemoryConfig, CommunicationConfig, ObservabilityConfig,
    SecurityConfig, Channel, BusConfig, CustomToolsConfig,
    PersistenceConfig,
)
from awp.runtime import (
    ToolRegistry, CodeExecutor, MessageBus,
    SecurityContext, CircuitBreaker, RateLimiter, AccessController,
    ObservabilityContext, Tracer, MetricsCollector, AuditTrail,
    StatePersistence,
)

print("All 30+ imports successful!")
print(f"Autonomy levels: {[level.name for level in AutonomyLevel]}")
print(f"Runtime components: ToolRegistry, CodeExecutor, MessageBus, SecurityContext, ObservabilityContext, StatePersistence")

All 30+ imports successful!
Autonomy levels: ['A0_PRESCRIBED', 'A1_ADAPTIVE', 'A2_DELEGATING', 'A3_SELF_TOOLING', 'A4_SELF_ORGANIZING']
Runtime components: ToolRegistry, CodeExecutor, MessageBus, SecurityContext, ObservabilityContext, StatePersistence


---

## 3. Build a Complete Workflow from Scratch

We will programmatically build a 3-agent data pipeline (collector -> analyzer -> reporter) that exercises **all 7 layers** plus security. Then we save it as YAML files.

In [4]:
import yaml

# Create a temporary directory for our workflow
workflow_dir = Path(tempfile.mkdtemp(prefix="awp-showcase"))
# Rename to a valid AWP workflow name (must match ^[a-z][a-z0-9_-]*[a-z0-9]$)
import shutil
clean_dir = workflow_dir.parent / "awp-showcase-demo"
if clean_dir.exists():
    shutil.rmtree(clean_dir)
workflow_dir.rename(clean_dir)
workflow_dir = clean_dir
agents_dir = workflow_dir / "agents"
print(f"Workflow directory: {workflow_dir}")

# ---------------------------------------------------------------
# Build the workflow manifest (all 7 layers + security)
# ---------------------------------------------------------------
workflow_yaml = {
    "awp": "1.0.0",
    "workflow": {
        "name": "awp-showcase-demo",  # R1: must match directory name
        "version": "1.0.0",
        "description": "Capstone data pipeline with all AWP features enabled",
        "author": "AWP Showcase",
        "tags": ["showcase", "a1", "all-features", "data-pipeline"],
    },

    # Layer 5: Orchestration -- DAG engine with 3 agents
    "orchestration": {
        "engine": "dag",
        "execution": {
            "mode": "sequential",
            "timeout": {"per_agent": 60, "total": 300},
            "max_parallel_agents": 3,
            "error_handling": {"default": "abort", "max_retries": 2},
        },
        "graph": [
            {
                "id": "collector",
                "agent": "collector",
                "depends_on": [],
                "share_output": ["raw_data", "data_quality"],
            },
            {
                "id": "analyzer",
                "agent": "analyzer",
                "depends_on": ["collector"],
                "share_output": ["statistics", "trends"],
            },
            {
                "id": "reporter",
                "agent": "reporter",
                "depends_on": ["analyzer"],
                "share_output": ["report"],
            },
        ],
    },

    # Layer 4 (State): Selective sharing
    "state": {
        "model": "shared_dict",
        "sharing": {"strategy": "selective"},
        "persistence": {
            "enabled": True,
            "path": "data/state",
            "format": "json",
            "snapshot_on_completion": True,
        },
    },

    # Layer 4 (Memory): Multi-tier memory
    "memory": {
        "enabled": True,
        "workspace_dir": "workspace",
        "long_term": {"enabled": True, "inject": True, "max_tokens": 2000},
        "daily_log": {"enabled": True, "auto_write": True, "retention_days": 30},
        "episodic": {"enabled": True, "max_entries": 100},
    },

    # Layer 3: Communication -- message bus with channels
    "communication": {
        "bus": {
            "type": "internal",
            "persistence": "run",
            "delivery": "at_least_once",
            "ordering": "fifo",
        },
        "channels": [
            {"name": "data_quality", "type": "topic", "description": "Data quality alerts"},
            {"name": "progress", "type": "broadcast", "description": "Pipeline progress events"},
            {"name": "results", "type": "direct", "description": "Final results delivery"},
        ],
    },

    # Layer 6: Observability -- tracing, metrics, audit
    "observability": {
        "logging": {
            "level": "INFO",
            "format": "json",
            "log_agent_io": True,
            "log_tool_calls": True,
        },
        "tracing": {
            "enabled": True,
            "exporter": "internal",
            "sample_rate": 1.0,
        },
        "metrics": {
            "enabled": True,
            "collector": "internal",
            "include": ["agent_duration", "tool_calls", "llm_tokens", "error_rate"],
        },
        "audit": {
            "enabled": True,
            "hash_chain": True,
            "include_events": ["agent_start", "agent_complete", "tool_call", "state_change", "error"],
        },
    },

    # Security: circuit breaker, rate limiter, access control
    "security": {
        "circuit_breaker": {
            "enabled": True,
            "failure_threshold": 3,
            "reset_timeout": 30,
        },
        "rate_limit": {
            "enabled": True,
            "max_calls_per_minute": 60,
            "per_agent": True,
        },
        "access_control": {
            "enabled": True,
            "default_policy": "allow",
            "rules": [
                {"agent": "reporter", "deny_tools": ["shell.execute"]},
            ],
        },
    },
}

# Write the workflow YAML
workflow_path = workflow_dir / "workflow.awp.yaml"
workflow_path.write_text(yaml.dump(workflow_yaml, default_flow_style=False, sort_keys=False))
print(f"Wrote: {workflow_path}")
print(f"Workflow name: {workflow_yaml['workflow']['name']}")
print(f"Layers configured: Manifest, Orchestration, State, Memory, Communication, Observability, Security")

Workflow directory: /tmp/awp-showcase-demo
Wrote: /tmp/awp-showcase-demo/workflow.awp.yaml
Workflow name: awp-showcase-demo
Layers configured: Manifest, Orchestration, State, Memory, Communication, Observability, Security


In [5]:
# ---------------------------------------------------------------
# Build agent definitions (Layer 1: Identity)
# ---------------------------------------------------------------

agent_definitions = {
    "collector": {
        "awp_agent": "1.0.0",
        "identity": {
            "id": "collector",
            "role": "data_collector",
            "description": "Collects and validates raw data from configured sources",
        },
        "model": {
            "name": "",
            "parameters": {"temperature": 0.1, "max_tokens": 2000},
        },
        "prompt": {
            "system": "You are a data collector agent. Gather, validate, and structure raw data.",
            "user_template": "Collect data for: {{task}}",
        },
        "output": {
            "format": "json",
            "contract": {
                "confidence": {"type": "number", "required": True},
                "raw_data": {"type": "object", "required": True},
                "data_quality": {"type": "object", "required": True},
            },
            "validation": {"mode": "strict", "on_invalid": "retry"},
        },
    },
    "analyzer": {
        "awp_agent": "1.0.0",
        "identity": {
            "id": "analyzer",
            "role": "data_analyzer",
            "description": "Analyzes data to extract statistics, trends, and insights",
        },
        "model": {
            "name": "",
            "parameters": {"temperature": 0.2, "max_tokens": 4000},
        },
        "prompt": {
            "system": "You are a data analyst. Compute statistics, find trends, and generate insights.",
            "user_template": "Analyze the following data: {{raw_data}}",
        },
        "output": {
            "format": "json",
            "contract": {
                "confidence": {"type": "number", "required": True},
                "statistics": {"type": "object", "required": True},
                "trends": {"type": "array", "required": True},
            },
            "validation": {"mode": "strict", "on_invalid": "retry"},
        },
    },
    "reporter": {
        "awp_agent": "1.0.0",
        "identity": {
            "id": "reporter",
            "role": "report_generator",
            "description": "Generates human-readable reports from analysis results",
        },
        "model": {
            "name": "",
            "parameters": {"temperature": 0.3, "max_tokens": 4000},
        },
        "prompt": {
            "system": "You are a report writer. Create clear, actionable reports from data analysis.",
            "user_template": "Write a report based on: {{statistics}} and {{trends}}",
        },
        "output": {
            "format": "json",
            "contract": {
                "confidence": {"type": "number", "required": True},
                "report": {"type": "string", "required": True},
            },
            "validation": {"mode": "strict", "on_invalid": "retry"},
        },
    },
}

# Write agent YAML files and output schemas
for agent_id, agent_def in agent_definitions.items():
    agent_path = agents_dir / agent_id
    agent_path.mkdir(parents=True, exist_ok=True)

    # Write agent.awp.yaml
    (agent_path / "agent.awp.yaml").write_text(
        yaml.dump(agent_def, default_flow_style=False, sort_keys=False)
    )

    # Write output_schema.json (R17: must have confidence field)
    schema_dir = agent_path / "workflow" / "output_schema"
    schema_dir.mkdir(parents=True, exist_ok=True)

    contract = agent_def["output"]["contract"]
    properties = {}
    required = []
    for field_name, field_spec in contract.items():
        prop = {"type": field_spec["type"]}
        if field_spec["type"] == "number" and field_name == "confidence":
            prop["minimum"] = 0.0
            prop["maximum"] = 1.0
        properties[field_name] = prop
        if field_spec.get("required", False):
            required.append(field_name)

    schema = {
        "type": "object",
        "properties": properties,
        "required": required,
    }
    (schema_dir / "output_schema.json").write_text(json.dumps(schema, indent=2))

    print(f"Created agent: {agent_id}  (fields: {list(contract.keys())})")

print(f"\nTotal agents: {len(agent_definitions)}")
print(f"Pipeline: collector -> analyzer -> reporter")

Created agent: collector  (fields: ['confidence', 'raw_data', 'data_quality'])
Created agent: analyzer  (fields: ['confidence', 'statistics', 'trends'])
Created agent: reporter  (fields: ['confidence', 'report'])

Total agents: 3
Pipeline: collector -> analyzer -> reporter


In [6]:
# Show the complete directory structure
def show_tree(path: Path, prefix: str = "", max_depth: int = 4, _depth: int = 0):
    if _depth >= max_depth:
        return
    items = sorted(path.iterdir()) if path.is_dir() else []
    for i, item in enumerate(items):
        is_last = i == len(items) - 1
        connector = "--- " if is_last else "|-- "
        print(f"{prefix}{connector}{item.name}")
        if item.is_dir():
            extension = "    " if is_last else "|   "
            show_tree(item, prefix + extension, max_depth, _depth + 1)

print(f"Workflow directory structure:")
print(workflow_dir.name + "/")
show_tree(workflow_dir)

Workflow directory structure:
awp-showcase-demo/
|-- agents
|   |-- analyzer
|   |   |-- agent.awp.yaml
|   |   --- workflow
|   |       --- output_schema
|   |-- collector
|   |   |-- agent.awp.yaml
|   |   --- workflow
|   |       --- output_schema
|   --- reporter
|       |-- agent.awp.yaml
|       --- workflow
|           --- output_schema
--- workflow.awp.yaml


---

## 4. Validate Everything

AWP provides four levels of validation:
1. **Schema validation** -- JSON Schema conformance for output schemas
2. **Graph validation** -- DAG structure (no cycles, valid refs, unique IDs)
3. **Rule validation** -- All 26 AWP rules (R1-R26)
4. **Compliance checking** -- Autonomy level assessment

In [7]:
# 4a. Parse the manifest we just created
manifest = parse_manifest(workflow_path)

print("=== Parsed Manifest ===")
print(f"  AWP version:   {manifest.awp}")
print(f"  Workflow:      {manifest.workflow.name}")
print(f"  Version:       {manifest.workflow.version}")
print(f"  Engine:        {manifest.orchestration.engine}")
print(f"  Graph nodes:   {len(manifest.orchestration.graph)}")
print(f"  State model:   {manifest.state.model}")
print(f"  State sharing: {manifest.state.sharing.strategy}")
print(f"  Memory:        {manifest.memory.enabled}")
print(f"  Observability: tracing={manifest.observability.tracing.enabled}, metrics={manifest.observability.metrics.enabled}, audit={manifest.observability.audit.enabled}")
print(f"  Security:      circuit_breaker={manifest.security.circuit_breaker.enabled}, rate_limit={manifest.security.rate_limit.enabled}")

=== Parsed Manifest ===
  AWP version:   1.0.0
  Workflow:      awp-showcase-demo
  Version:       1.0.0
  Engine:        dag
  Graph nodes:   3
  State model:   shared_dict
  State sharing: selective
  Memory:        True
  Observability: tracing=True, metrics=True, audit=True
  Security:      circuit_breaker=True, rate_limit=True


In [8]:
# 4b. Validate all output schemas
print("=== Schema Validation (R17: confidence field required) ===")
all_schemas_valid = True
for agent_id in agent_definitions:
    schema_path = agents_dir / agent_id / "workflow" / "output_schema" / "output_schema.json"
    result = validate_schema(schema_path)
    status = "PASS" if result.valid else "FAIL"
    print(f"  [{status}] {agent_id}: valid={result.valid}, errors={result.errors}")
    if not result.valid:
        all_schemas_valid = False

print(f"\nAll schemas valid: {all_schemas_valid}")

=== Schema Validation (R17: confidence field required) ===
  [PASS] collector: valid=True, errors=[]
  [PASS] analyzer: valid=True, errors=[]
  [PASS] reporter: valid=True, errors=[]

All schemas valid: True


In [9]:
# 4b. Graph validation (R2, R6, R7)
result = validate_graph(manifest.orchestration)
print("=== Graph Validation (R2, R6, R7: unique IDs, valid refs, no cycles) ===")
print(f"  Valid:    {result.valid}")
print(f"  Errors:   {result.errors}")
print(f"  Warnings: {result.warnings}")

# Show the execution order
print("\nExecution order (topological):")
for node in manifest.orchestration.graph:
    deps = node.depends_on if node.depends_on else ["(root)"]
    print(f"  {node.id:12s} <- {str(deps):30s} shares: {node.share_output}")

=== Graph Validation (R2, R6, R7: unique IDs, valid refs, no cycles) ===
  Valid:    True
  Errors:   []
  Warnings: []

Execution order (topological):
  collector    <- ['(root)']                     shares: ['raw_data', 'data_quality']
  analyzer     <- ['collector']                  shares: ['statistics', 'trends']
  reporter     <- ['analyzer']                   shares: ['report']


In [10]:
# 4d. Validate all rules (R1-R26) and check compliance
# Load agents
agents = {}
for agent_id in agent_definitions:
    agents[agent_id] = parse_agent(agents_dir / agent_id / "agent.awp.yaml")

print(f"Loaded {len(agents)} agents: {list(agents.keys())}")

# Rule validation
rules_result = validate_rules(manifest, agents, workflow_path=workflow_dir)
print(f"\n=== Rule Validation (R1-R26) ===")
print(f"  Valid:  {rules_result.valid}")
if rules_result.errors:
    print(f"  Errors ({len(rules_result.errors)}):")
    for e in rules_result.errors:
        print(f"    - {e}")
else:
    print("  No errors -- all rules pass!")
if rules_result.warnings:
    print(f"  Warnings ({len(rules_result.warnings)}):")
    for w in rules_result.warnings:
        print(f"    - {w}")

Loaded 3 agents: ['collector', 'analyzer', 'reporter']

=== Rule Validation (R1-R26) ===
  Valid:  False
  Errors (12):
    - R11: Agent 'collector' missing agent.py
    - R11: Agent 'collector' missing SYSTEM_PROMPT.md
    - R11: Agent 'collector' missing 00_INTRO.md
    - R11: Agent 'collector' missing output_schema_desc.json
    - R11: Agent 'analyzer' missing agent.py
    - R11: Agent 'analyzer' missing SYSTEM_PROMPT.md
    - R11: Agent 'analyzer' missing 00_INTRO.md
    - R11: Agent 'analyzer' missing output_schema_desc.json
    - R11: Agent 'reporter' missing agent.py
    - R11: Agent 'reporter' missing SYSTEM_PROMPT.md
    - R11: Agent 'reporter' missing 00_INTRO.md
    - R11: Agent 'reporter' missing output_schema_desc.json


In [11]:
# 4e. Check compliance / autonomy level
compliance = check_compliance(
    manifest, agents,
    workflow_path=workflow_dir,
    target_level=AutonomyLevel.A4_SELF_ORGANIZING,
)

print("=== Compliance Check ===")
print(f"  Achieved level:  {compliance.level.name} ({compliance.level_name})")
print(f"  Max achievable:  {compliance.max_achievable.name}")
print(f"  Compliant:       {compliance.compliant}")
if compliance.checks:
    print(f"  Checks:")
    for k, v in compliance.checks.items():
        status = "PASS" if v else "FAIL"
        print(f"    [{status}] {k}")

print(f"\nThis workflow achieves A1 (Adaptive) because it has a multi-agent DAG.")
print(f"To reach A2, it would need a delegation loop engine with budget constraints.")

=== Compliance Check ===
  Achieved level:  A1_ADAPTIVE (Adaptive)
  Max achievable:  A4_SELF_ORGANIZING
  Compliant:       False
  Checks:
    [PASS] manifest_present
    [PASS] awp_version_set
    [PASS] workflow_name_valid
    [PASS] at_least_one_agent
    [PASS] agent_collector_has_contract
    [PASS] agent_analyzer_has_contract
    [PASS] agent_reporter_has_contract
    [PASS] has_orchestration
    [PASS] has_graph_or_delegation
    [FAIL] has_adaptive_features
    [PASS] multi_agent_dag
    [FAIL] delegation_loop_engine
    [FAIL] delegation_loop_config
    [FAIL] dynamic_tools_enabled
    [FAIL] has_tool_creator_agent
    [FAIL] delegation_allows_tool_creation

This workflow achieves A1 (Adaptive) because it has a multi-agent DAG.
To reach A2, it would need a delegation loop engine with budget constraints.


---

## 5. Run with AgentWorkflow (Data-Driven)

`AgentWorkflow` is AWP's highest-level API. Pass arbitrary inputs + a task description, and the A4 delegation loop handles the rest. This cell requires a working LLM provider.

**Inputs** can be inline Python objects (DataFrame, dict, list, str, etc.) or
**`Source` objects** for remote/offline data (`Source.url()`, `Source.sql()`,
`Source.s3()`, `Source.glob()`, `Source.api()`, `Source.base64()`). Sources
are resolved in parallel before the workflow starts.

In [ ]:
import pandas as pd
import base64
from awp.data import AgentWorkflow, Source

# Create sample data inline
df = pd.DataFrame({
    "date": pd.date_range("2024-01-01", periods=30, freq="D"),
    "temperature": [20 + i * 0.5 + (i % 7) * 2 for i in range(30)],
    "humidity": [60 - i * 0.3 + (i % 5) * 3 for i in range(30)],
    "city": ["Berlin", "Munich", "Hamburg"] * 10,
})

# Encode supplementary config as base64 (demonstrates Source.base64)
config_b64 = base64.b64encode(
    b'{"focus_cities": ["Berlin", "Munich"], "unit": "celsius"}'
).decode()

print(f"DataFrame: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
print(f"Source config: base64-encoded JSON ({len(config_b64)} chars)")

In [ ]:
# Run the full delegation loop workflow with mixed inline + Source inputs
result = AgentWorkflow(
    inputs={
        "weather_data": df,
        # Source: decoded at resolve time, agents see a JSON string
        "analysis_config": Source.base64(config_b64, format="text"),
    },
    task=(
        "Analyze the weather data: compare cities, find trends, "
        "calculate averages per city, and identify the warmest/coldest days. "
        "Use the analysis_config for focus cities and temperature unit."
    ),
    model=MODEL,
    max_loops=5,
    max_wall_time=180,
    max_total_tokens=200_000,
    code_mode=True,
    tool_creation=True,
    output_dir="./output_full_showcase",
    verbose=True,
).run()

print(f"\n{'='*60}")
print(f"Status: {result['status']}")

---

## 6. Inspect All Outputs

After a workflow run, AWP produces structured outputs: the result dict, metadata, and any artifacts (files, charts, reports).

In [14]:
# 6a. Result overview
print("=== Result Status ===")
print(f"  Status:  {result.get('status', 'unknown')}")
print()

print("=== Metadata ===")
metadata = result.get("metadata", {})
for k, v in metadata.items():
    print(f"  {k}: {v}")
print()

print("=== Result (first 2000 chars) ===")
result_str = json.dumps(result.get("result", {}), indent=2, default=str)
print(result_str[:2000])
if len(result_str) > 2000:
    print(f"  ... ({len(result_str)} total characters)")

=== Result Status ===
  Status:  error

=== Metadata ===
  loops: 1
  tokens_used: 0
  wall_time: 0.25
  workers_spawned: 0
  tool_calls: 0
  workspace: /home/shumway/projects/agent-workflow-protocol/output_full_showcase

=== Result (first 2000 chars) ===
{
  "error": "Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401",
  "partial_result": {},
  "confidence": 0.0
}


In [15]:
# 6b. List artifacts
print("=== Artifacts ===")
artifacts = result.get("artifacts", [])
if artifacts:
    for a in artifacts:
        if isinstance(a, dict):
            print(f"  - {a.get('name', 'unnamed')}: {a.get('path', 'no path')}")
        else:
            print(f"  - {a}")
else:
    print("  No artifacts generated (this is normal for short runs).")

# 6c. Show workspace structure if output directory exists
output_path = Path("./output_full_showcase")
if output_path.exists():
    print(f"\n=== Workspace Structure ===")
    show_tree(output_path, max_depth=3)
    
    # Read any generated files
    for f in output_path.rglob("*"):
        if f.is_file() and f.stat().st_size < 5000:
            print(f"\n--- {f.relative_to(output_path)} ---")
            try:
                content = f.read_text()
                print(content[:1000])
                if len(content) > 1000:
                    print(f"... ({len(content)} chars total)")
            except UnicodeDecodeError:
                print(f"(binary file, {f.stat().st_size} bytes)")
else:
    print(f"\nOutput directory not found (workflow may not have completed).")

=== Artifacts ===
  No artifacts generated (this is normal for short runs).

=== Workspace Structure ===
|-- agents
|   --- manager
|       |-- agent.awp.yaml
|       --- system_prompt.md
|-- output
--- workspace
    |-- input_manifest.json
    |-- inputs
    |   --- weather_data.csv
    --- runs
        --- 2026-03-28_03-20-53_ca5a00c2

--- workspace/input_manifest.json ---
{
  "weather_data": {
    "type": "dataframe",
    "key": "weather_data",
    "workspace_path": "inputs/weather_data.csv",
    "schema": {
      "shape": [
        30,
        4
      ],
      "columns": [
        "date",
        "temperature",
        "humidity",
        "city"
      ],
      "dtypes": {
        "date": "datetime64[us]",
        "temperature": "float64",
        "humidity": "float64",
        "city": "str"
      },
      "head": [
        {
          "date": "2024-01-01 00:00:00",
          "temperature": 20.0,
          "humidity": 60.0,
          "city": "Berlin"
        },
        {
          "

---

## 7. Runtime Components Demo (All Together)

This section shows all runtime components wired together in one integrated example. No LLM calls needed -- this demonstrates the infrastructure layer.

In [16]:
import uuid

# Create a temporary workspace for the demo
demo_dir = Path(tempfile.mkdtemp(prefix="awp_runtime_demo_"))
run_id = f"demo_{uuid.uuid4().hex[:8]}"

print(f"Demo workspace: {demo_dir}")
print(f"Run ID: {run_id}")
print()

# ---------------------------------------------------------------
# 1. Observability -- set up tracing, metrics, audit
# ---------------------------------------------------------------
tracer = Tracer(output_dir=demo_dir / "data" / "traces", run_id=run_id)
metrics = MetricsCollector(output_dir=demo_dir / "data" / "metrics", run_id=run_id)
audit = AuditTrail(output_dir=demo_dir / "data" / "audit", run_id=run_id)

obs = ObservabilityContext(tracer=tracer, metrics=metrics, audit=audit)
print("[1] Observability: Tracer + MetricsCollector + AuditTrail initialized")

# ---------------------------------------------------------------
# 2. Security -- circuit breaker, rate limiter, access control
# ---------------------------------------------------------------
cb = CircuitBreaker(failure_threshold=3, reset_timeout=30.0)
rl = RateLimiter(max_calls_per_minute=60, per_agent=True)
ac = AccessController(
    default_policy="allow",
    rules=[{"agent": "reporter", "deny_tools": ["shell.execute", "file.write"]}],
)
security = SecurityContext(circuit_breaker=cb, rate_limiter=rl, access_controller=ac)
print("[2] Security: CircuitBreaker + RateLimiter + AccessController initialized")

# ---------------------------------------------------------------
# 3. Message Bus -- inter-agent communication
# ---------------------------------------------------------------
bus = MessageBus()
print("[3] MessageBus: ready for inter-agent messaging")

# ---------------------------------------------------------------
# 4. State Persistence -- per-agent checkpoints
# ---------------------------------------------------------------
state_store = StatePersistence(output_dir=demo_dir / "data" / "state")
print("[4] StatePersistence: JSON-based checkpoint store ready")

# ---------------------------------------------------------------
# 5. Tool Registry -- built-in tools + code executor
# ---------------------------------------------------------------
registry = ToolRegistry(workflow_dir=demo_dir)
executor = CodeExecutor(max_timeout=10, working_dir=demo_dir)
registry.set_code_executor(executor)
registry.set_message_bus(bus)
registry.set_security_context(security)
print("[5] ToolRegistry: built-in tools + CodeExecutor + MessageBus + Security wired")

print(f"\nAll 5 runtime subsystems initialized and interconnected.")

Demo workspace: /tmp/awp_runtime_demo_i4aon4vi
Run ID: demo_0b32b09a

[1] Observability: Tracer + MetricsCollector + AuditTrail initialized
[2] Security: CircuitBreaker + RateLimiter + AccessController initialized
[3] MessageBus: ready for inter-agent messaging
[4] StatePersistence: JSON-based checkpoint store ready
[5] ToolRegistry: built-in tools + CodeExecutor + MessageBus + Security wired

All 5 runtime subsystems initialized and interconnected.


In [17]:
# ---------------------------------------------------------------
# Simulate a complete pipeline execution using all components
# ---------------------------------------------------------------

# Start a workflow-level trace span
workflow_span = tracer.start_span("workflow_run", attributes={"run_id": run_id})
audit.record("workflow_start", details={"run_id": run_id, "agents": ["collector", "analyzer", "reporter"]})

# --- Agent 1: Collector ---
print("=== Agent: collector ===")
collector_span = tracer.start_span("agent_collector", parent_id=workflow_span)

# Security check
assert cb.check(), "Circuit breaker should allow calls"
assert rl.check("collector"), "Rate limiter should allow calls"
assert ac.is_allowed("collector", "code.execute"), "Collector should be allowed code.execute"
rl.record("collector")
print("  Security: all checks passed")

# Execute code via the tool registry
code_result = registry.call("code.execute", {
    "code": "import json; data = {'temp': [20,22,25], 'city': 'Berlin'}; print(json.dumps(data))"
})
print(f"  Code execution: ok={code_result.get('ok')}, output={code_result.get('data', {}).get('stdout', '')[:100]}")
cb.record_success()
metrics.increment("tool_calls", labels={"agent": "collector", "tool": "code.execute"})

# Save state checkpoint
collector_state = {"raw_data": {"temp": [20, 22, 25], "city": "Berlin"}, "data_quality": {"completeness": 1.0}}
state_store.save_checkpoint("collector", collector_state)
print(f"  State checkpoint saved")

# Send results to analyzer via message bus
msg_id = bus.send("collector", "analyzer", collector_state, channel="data_quality")
print(f"  Message sent to analyzer: {msg_id[:16]}...")

# Broadcast progress
bus.broadcast("collector", {"stage": "collection", "status": "complete"}, channel="progress")
print(f"  Progress broadcast sent")

tracer.end_span(collector_span, status="ok")
audit.record("agent_complete", agent_id="collector", details={"confidence": 0.95})
metrics.histogram("agent_duration", 1.23, labels={"agent": "collector"})
print(f"  Span closed, audit logged, metrics recorded")

DEBUG:awp.runtime.state_persistence:State checkpoint saved: /tmp/awp_runtime_demo_i4aon4vi/data/state/collector.json


INFO:awp.runtime.message_bus:Bus: collector → analyzer on channel 'data_quality'


INFO:awp.runtime.message_bus:Bus: broadcast from collector on channel 'progress'


=== Agent: collector ===
  Security: all checks passed
  Code execution: ok=True, output={"temp": [20, 22, 25], "city": "Berlin"}

  State checkpoint saved
  Message sent to analyzer: 498fbe3b-34b6-47...
  Progress broadcast sent
  Span closed, audit logged, metrics recorded


In [18]:
# --- Agent 2: Analyzer ---
print("=== Agent: analyzer ===")
analyzer_span = tracer.start_span("agent_analyzer", parent_id=workflow_span)

# Receive messages
inbox = bus.list_messages("analyzer")
print(f"  Received {len(inbox)} message(s) from bus")
if inbox:
    received_data = inbox[0].get("content", {})
    print(f"  Data from collector: {list(received_data.keys())}")

# Security check
assert cb.check() and rl.check("analyzer")
rl.record("analyzer")
print("  Security: all checks passed")

# Run analysis via code executor
analysis_code = """
import json, statistics
data = [20, 22, 25]
result = {
    'mean': statistics.mean(data),
    'stdev': round(statistics.stdev(data), 2),
    'min': min(data),
    'max': max(data),
    'trend': 'increasing'
}
print(json.dumps(result))
"""
analysis_result = registry.call("code.execute", {"code": analysis_code})
print(f"  Analysis result: {analysis_result.get('data', {}).get('stdout', '')[:200]}")
cb.record_success()
metrics.increment("tool_calls", labels={"agent": "analyzer", "tool": "code.execute"})

# Save state
analyzer_state = {"statistics": {"mean": 22.33, "stdev": 2.52}, "trends": ["increasing"]}
state_store.save_checkpoint("analyzer", analyzer_state)

# Send to reporter
bus.send("analyzer", "reporter", analyzer_state, channel="results")
bus.broadcast("analyzer", {"stage": "analysis", "status": "complete"}, channel="progress")

tracer.end_span(analyzer_span, status="ok")
audit.record("agent_complete", agent_id="analyzer", details={"confidence": 0.88})
metrics.histogram("agent_duration", 2.45, labels={"agent": "analyzer"})
print(f"  Analyzer complete: state saved, messages sent, observability logged")

DEBUG:awp.runtime.state_persistence:State checkpoint saved: /tmp/awp_runtime_demo_i4aon4vi/data/state/analyzer.json


INFO:awp.runtime.message_bus:Bus: analyzer → reporter on channel 'results'


INFO:awp.runtime.message_bus:Bus: broadcast from analyzer on channel 'progress'


=== Agent: analyzer ===
  Received 2 message(s) from bus
  Data from collector: ['stage', 'status']
  Security: all checks passed
  Analysis result: {"mean": 22.333333333333332, "stdev": 2.52, "min": 20, "max": 25, "trend": "increasing"}

  Analyzer complete: state saved, messages sent, observability logged


In [19]:
# --- Agent 3: Reporter ---
print("=== Agent: reporter ===")
reporter_span = tracer.start_span("agent_reporter", parent_id=workflow_span)

# Receive messages
inbox = bus.list_messages("reporter")
print(f"  Received {len(inbox)} message(s)")

# Security: reporter is denied shell.execute
assert ac.is_allowed("reporter", "code.execute"), "Reporter can use code.execute"
assert not ac.is_allowed("reporter", "shell.execute"), "Reporter is denied shell.execute"
assert not ac.is_allowed("reporter", "file.write"), "Reporter is denied file.write"
print("  Security: code.execute=ALLOWED, shell.execute=DENIED, file.write=DENIED")

# Save final state
reporter_state = {"report": "Temperature analysis: mean=22.33C, trend=increasing, city=Berlin"}
state_store.save_checkpoint("reporter", reporter_state)

# Save final workflow state
final_state = {
    "collector": collector_state,
    "analyzer": analyzer_state,
    "reporter": reporter_state,
}
state_store.save_final(final_state)
print(f"  Final state saved with all 3 agent outputs")

# Close everything
tracer.end_span(reporter_span, status="ok")
tracer.end_span(workflow_span, status="ok")
audit.record("agent_complete", agent_id="reporter", details={"confidence": 0.92})
audit.record("workflow_complete", details={"run_id": run_id, "status": "success"})
metrics.histogram("agent_duration", 0.87, labels={"agent": "reporter"})
metrics.increment("workflow_completions")

# Flush all observability data to disk
trace_path = tracer.flush()
metrics_path = metrics.flush()
audit_path = audit.flush()

print(f"\n=== Observability Output ===")
print(f"  Traces: {trace_path}")
print(f"  Metrics: {metrics_path}")
print(f"  Audit: {audit_path}")

DEBUG:awp.runtime.state_persistence:State checkpoint saved: /tmp/awp_runtime_demo_i4aon4vi/data/state/reporter.json


INFO:awp.runtime.state_persistence:Final state saved: /tmp/awp_runtime_demo_i4aon4vi/data/state/final.json


INFO:awp.runtime.observability:Flushed 4 trace spans to /tmp/awp_runtime_demo_i4aon4vi/data/traces/demo_0b32b09a.jsonl


INFO:awp.runtime.observability:Flushed metrics to /tmp/awp_runtime_demo_i4aon4vi/data/metrics/demo_0b32b09a.json


INFO:awp.runtime.observability:Flushed 5 audit entries to /tmp/awp_runtime_demo_i4aon4vi/data/audit/demo_0b32b09a.jsonl


=== Agent: reporter ===
  Received 3 message(s)
  Security: code.execute=ALLOWED, shell.execute=DENIED, file.write=DENIED
  Final state saved with all 3 agent outputs

=== Observability Output ===
  Traces: /tmp/awp_runtime_demo_i4aon4vi/data/traces/demo_0b32b09a.jsonl
  Metrics: /tmp/awp_runtime_demo_i4aon4vi/data/metrics/demo_0b32b09a.json
  Audit: /tmp/awp_runtime_demo_i4aon4vi/data/audit/demo_0b32b09a.jsonl


In [20]:
# Inspect the observability output files
print("=== Trace Spans ===")
if trace_path and trace_path.exists():
    for line in trace_path.read_text().strip().split("\n"):
        span = json.loads(line)
        print(f"  {span['name']:25s}  duration={span.get('duration_ms', '?')}ms  status={span.get('status', '?')}")

print("\n=== Metrics ===")
if metrics_path and metrics_path.exists():
    metrics_data = json.loads(metrics_path.read_text())
    for k, v in metrics_data.get("counters", {}).items():
        print(f"  Counter: {k} = {v}")
    for k, v in metrics_data.get("histograms", {}).items():
        print(f"  Histogram: {k} -> count={v['count']}, min={v['min']}, max={v['max']}")

print("\n=== Audit Trail (hash-chained) ===")
if audit_path and audit_path.exists():
    audit_entries = []
    for line in audit_path.read_text().strip().split("\n"):
        entry = json.loads(line)
        audit_entries.append(entry)
        print(f"  seq={entry.get('seq', '?'):2}  {entry.get('event_type', '?'):25s}  hash={entry.get('hash', '?')[:16]}...")

    # Verify the hash chain integrity
    chain_valid = AuditTrail.verify_chain(audit_entries)
    print(f"\n  Hash chain integrity: {'VALID' if chain_valid else 'BROKEN'}")

print("\n=== State Checkpoints ===")
for agent_id in ["collector", "analyzer", "reporter"]:
    loaded = state_store.load_checkpoint(agent_id)
    if loaded:
        print(f"  {agent_id}: {list(loaded.keys())}")

final = state_store.load_final()
if final:
    print(f"  final: {list(final.keys())}")

=== Trace Spans ===
  agent_collector            duration=14.91ms  status=ok
  agent_analyzer             duration=16.17ms  status=ok
  agent_reporter             duration=1.04ms  status=ok
  workflow_run               duration=39.6ms  status=ok

=== Metrics ===
  Counter: tool_calls{agent=collector,tool=code.execute} = 1.0
  Counter: tool_calls{agent=analyzer,tool=code.execute} = 1.0
  Counter: workflow_completions = 1.0
  Histogram: agent_duration{agent=collector} -> count=1, min=1.23, max=1.23
  Histogram: agent_duration{agent=analyzer} -> count=1, min=2.45, max=2.45
  Histogram: agent_duration{agent=reporter} -> count=1, min=0.87, max=0.87

=== Audit Trail (hash-chained) ===
  seq= 1  workflow_start             hash=bc9284f71b4cb71c...
  seq= 2  agent_complete             hash=35e09e37808df0f7...
  seq= 3  agent_complete             hash=1cd669ec0bb5d2b6...
  seq= 4  agent_complete             hash=34b46970aa879f20...
  seq= 5  workflow_complete          hash=b81ea61b53662c43...

 

In [21]:
# Demonstrate circuit breaker transitions
print("=== Circuit Breaker Demo ===")
demo_cb = CircuitBreaker(failure_threshold=3, reset_timeout=1.0)

print(f"  Initial state: {demo_cb.state}")
print(f"  Can call: {demo_cb.check()}")

# Simulate 3 failures to trip it
for i in range(3):
    demo_cb.record_failure()
    print(f"  After failure {i+1}: state={demo_cb.state}")

print(f"  Can call while open: {demo_cb.check()}")

# Wait for reset
import time
time.sleep(1.1)
print(f"  After 1.1s wait: state={demo_cb.state}")
print(f"  Can call (half-open): {demo_cb.check()}")

# Record success to close
demo_cb.record_success()
print(f"  After success: state={demo_cb.state}")
print(f"  Can call: {demo_cb.check()}")

=== Circuit Breaker Demo ===
  Initial state: closed
  Can call: True
  After failure 1: state=closed
  After failure 2: state=closed
  After failure 3: state=open
  Can call while open: False


INFO:awp.runtime.security:Circuit breaker: open → half_open (timeout elapsed)


INFO:awp.runtime.security:Circuit breaker: half_open → closed


  After 1.1s wait: state=half_open
  Can call (half-open): True
  After success: state=closed
  Can call: True


In [22]:
# Demonstrate message bus patterns
print("=== Message Bus Patterns ===")
demo_bus = MessageBus()

# Direct message
msg1 = demo_bus.send("agent_a", "agent_b", {"task": "process data"}, channel="tasks")
print(f"  Direct:    agent_a -> agent_b (msg={msg1[:12]}...)")

# Broadcast
msg2 = demo_bus.broadcast("agent_a", {"alert": "data ready"}, channel="alerts")
print(f"  Broadcast: agent_a -> * (msg={msg2[:12]}...)")

# Channel-based
msg3 = demo_bus.send("agent_b", "agent_c", {"result": 42}, channel="results")
print(f"  Channel:   agent_b -> agent_c on 'results' (msg={msg3[:12]}...)")

# Receive messages using list_messages
b_inbox = demo_bus.list_messages("agent_b")
c_inbox = demo_bus.list_messages("agent_c")

print(f"\n  agent_b inbox: {len(b_inbox)} message(s)")
for m in b_inbox:
    print(f"    from={m['from']}, channel={m['channel']}, content={m['content']}")

print(f"  agent_c inbox: {len(c_inbox)} message(s)")
for m in c_inbox:
    print(f"    from={m['from']}, channel={m['channel']}, content={m['content']}")

# Channel history
tasks_history = demo_bus.get_channel_messages("tasks")
results_history = demo_bus.get_channel_messages("results")
alerts_history = demo_bus.get_channel_messages("alerts")
print(f"\n  Channel messages: tasks={len(tasks_history)}, results={len(results_history)}, alerts={len(alerts_history)}")

# Filtered queries: messages for agent_b from agent_a only
filtered = demo_bus.list_messages("agent_b", from_agent="agent_a")
print(f"  agent_b messages from agent_a: {len(filtered)}")

# Filter by channel
channel_filtered = demo_bus.list_messages("agent_b", channel="tasks")
print(f"  agent_b messages on 'tasks' channel: {len(channel_filtered)}")

INFO:awp.runtime.message_bus:Bus: agent_a → agent_b on channel 'tasks'


INFO:awp.runtime.message_bus:Bus: broadcast from agent_a on channel 'alerts'


INFO:awp.runtime.message_bus:Bus: agent_b → agent_c on channel 'results'


=== Message Bus Patterns ===
  Direct:    agent_a -> agent_b (msg=a5567c30-767...)
  Broadcast: agent_a -> * (msg=c9878dae-84c...)
  Channel:   agent_b -> agent_c on 'results' (msg=9c9df89a-301...)

  agent_b inbox: 2 message(s)
    from=agent_a, channel=alerts, content={'alert': 'data ready'}
    from=agent_a, channel=tasks, content={'task': 'process data'}
  agent_c inbox: 2 message(s)
    from=agent_b, channel=results, content={'result': 42}
    from=agent_a, channel=alerts, content={'alert': 'data ready'}

  Channel messages: tasks=1, results=1, alerts=1
  agent_b messages from agent_a: 2
  agent_b messages on 'tasks' channel: 1


---

## 8. Autonomy Levels Side by Side

AWP defines 5 autonomy levels (A0-A4). Let us load real examples from each level, check their compliance, and compare features.

In [23]:
# Helper to load agents from a workflow directory
def load_agents(workflow_dir: Path) -> dict:
    agents = {}
    agents_dir = workflow_dir / "agents"
    if not agents_dir.exists():
        return agents
    for agent_dir in sorted(agents_dir.iterdir()):
        agent_yaml = agent_dir / "agent.awp.yaml"
        if agent_yaml.exists():
            try:
                agent = parse_agent(agent_yaml)
                agents[agent.identity.id] = agent
            except Exception as e:
                pass  # Skip agents that fail to parse
    return agents


# Define the examples to compare
examples = [
    ("01-hello-world", "A0 Prescribed"),
    ("02-research-pipeline", "A1 Adaptive"),
    ("05-observable-analytics", "A1 + Observability"),
    ("08-delegation-loop", "A2 Delegating"),
    ("09-recursive-delegation", "A2+ Recursive"),
    ("12-full-autonomy-test", "A3/A4 Self-Organizing"),
]

results_table = []
for example_name, label in examples:
    ex_dir = Path(f"examples/{example_name}")
    if not ex_dir.exists():
        print(f"  Skipping {example_name} (not found)")
        continue

    try:
        m = parse_manifest(ex_dir / "workflow.awp.yaml")
        a = load_agents(ex_dir)
        c = check_compliance(m, a, workflow_path=ex_dir, target_level=AutonomyLevel.A4_SELF_ORGANIZING)

        # Detect features
        has_dag = bool(m.orchestration and m.orchestration.graph)
        has_delegation = bool(m.orchestration and m.orchestration.delegation_loop)
        has_memory = bool(m.memory and m.memory.enabled)
        has_obs = bool(m.observability and (m.observability.tracing.enabled or m.observability.metrics.enabled))
        has_security = bool(m.security and m.security.circuit_breaker.enabled)
        has_dynamic = bool(m.dynamic_tools and m.dynamic_tools.enabled)
        has_comm = bool(m.communication and m.communication.channels)

        results_table.append({
            "example": example_name,
            "label": label,
            "agents": len(a),
            "level": c.level.name,
            "level_name": c.level_name,
            "dag": has_dag,
            "delegation": has_delegation,
            "memory": has_memory,
            "observability": has_obs,
            "security": has_security,
            "dynamic_tools": has_dynamic,
            "communication": has_comm,
        })
        print(f"  Loaded: {example_name:30s} -> {c.level.name} ({c.level_name})")
    except Exception as e:
        print(f"  Error loading {example_name}: {e}")

print(f"\nLoaded {len(results_table)} examples for comparison.")

  Loaded: 01-hello-world                 -> A0_PRESCRIBED (Prescribed)
  Loaded: 02-research-pipeline           -> A1_ADAPTIVE (Adaptive)
  Loaded: 05-observable-analytics        -> A1_ADAPTIVE (Adaptive)
  Loaded: 08-delegation-loop             -> A3_SELF_TOOLING (Self-Tooling)
  Loaded: 09-recursive-delegation        -> A3_SELF_TOOLING (Self-Tooling)
  Loaded: 12-full-autonomy-test          -> A3_SELF_TOOLING (Self-Tooling)

Loaded 6 examples for comparison.


In [24]:
# Build a comparison table
def bool_mark(v: bool) -> str:
    return "Yes" if v else "-"

print("=" * 120)
print(f"{'Example':<30s} {'Level':<28s} {'#Ag':>4s} {'DAG':>5s} {'Deleg':>6s} {'Mem':>5s} {'Obs':>5s} {'Sec':>5s} {'Dyn':>5s} {'Comm':>5s}")
print("-" * 120)
for r in results_table:
    print(
        f"{r['example']:<30s} {r['level'] + ' ' + r['level_name']:<28s} {r['agents']:>4d}"
        f" {bool_mark(r['dag']):>5s} {bool_mark(r['delegation']):>6s}"
        f" {bool_mark(r['memory']):>5s} {bool_mark(r['observability']):>5s}"
        f" {bool_mark(r['security']):>5s} {bool_mark(r['dynamic_tools']):>5s}"
        f" {bool_mark(r['communication']):>5s}"
    )
print("=" * 120)

print("\nKey:")
print("  #Ag = Number of agents")
print("  DAG = Has a DAG execution graph")
print("  Deleg = Uses delegation loop engine")
print("  Mem = Memory enabled")
print("  Obs = Observability (tracing/metrics) enabled")
print("  Sec = Security (circuit breaker) enabled")
print("  Dyn = Dynamic tool creation enabled")
print("  Comm = Has communication channels defined")

Example                        Level                         #Ag   DAG  Deleg   Mem   Obs   Sec   Dyn  Comm
------------------------------------------------------------------------------------------------------------------------
01-hello-world                 A0_PRESCRIBED Prescribed        1   Yes      -     -     -     -     -     -
02-research-pipeline           A1_ADAPTIVE Adaptive            3   Yes      -     -     -     -     -     -
05-observable-analytics        A1_ADAPTIVE Adaptive            3   Yes      -   Yes   Yes     -     -     -
08-delegation-loop             A3_SELF_TOOLING Self-Tooling    1     -    Yes     -     -     -     -     -
09-recursive-delegation        A3_SELF_TOOLING Self-Tooling    1     -    Yes     -     -     -     -     -
12-full-autonomy-test          A3_SELF_TOOLING Self-Tooling    1     -    Yes     -     -     -   Yes     -

Key:
  #Ag = Number of agents
  DAG = Has a DAG execution graph
  Deleg = Uses delegation loop engine
  Mem = Memory enabl

In [25]:
# Show what each autonomy level adds
print("=== What Each Autonomy Level Adds ===")
print()
level_features = [
    ("A0 Prescribed", [
        "Workflow manifest with metadata",
        "At least 1 agent with identity, model config, prompt",
        "Output contract with confidence field (R17)",
        "JSON Schema validation for outputs",
        "Basic state sharing (full strategy)",
    ]),
    ("A1 Adaptive", [
        "Everything in A0, plus:",
        "Multi-agent DAG with dependency ordering",
        "Selective state sharing between agents",
        "Conditional execution (on_failure, retry)",
        "Parallel execution (fan-out)",
        "Communication channels (message bus)",
    ]),
    ("A2 Delegating", [
        "Everything in A1, plus:",
        "Delegation loop engine (manager-worker pattern)",
        "Budget enforcement (max_loops, max_workers, max_tokens, max_wall_time)",
        "Dynamic worker spawning by manager",
        "Validation gates (deterministic + LLM-based)",
        "Stall detection and termination",
    ]),
    ("A3 Self-Tooling", [
        "Everything in A2, plus:",
        "dynamic_tools.enabled = true",
        "Agents create tools at runtime (code.execute + tool registration)",
        "Safety envelope: namespace restrictions, code review option",
        "Tool persistence across iterations",
    ]),
    ("A4 Self-Organizing", [
        "Everything in A3, plus:",
        "Recursive delegation (max_depth > 1)",
        "Budget distribution across sub-workflows",
        "Full observability required (tracing + metrics + audit)",
        "Self-organizing agent topologies",
    ]),
]

for level_name, features in level_features:
    print(f"--- {level_name} ---")
    for f in features:
        print(f"  * {f}")
    print()

=== What Each Autonomy Level Adds ===

--- A0 Prescribed ---
  * Workflow manifest with metadata
  * At least 1 agent with identity, model config, prompt
  * Output contract with confidence field (R17)
  * JSON Schema validation for outputs
  * Basic state sharing (full strategy)

--- A1 Adaptive ---
  * Everything in A0, plus:
  * Multi-agent DAG with dependency ordering
  * Selective state sharing between agents
  * Conditional execution (on_failure, retry)
  * Parallel execution (fan-out)
  * Communication channels (message bus)

--- A2 Delegating ---
  * Everything in A1, plus:
  * Delegation loop engine (manager-worker pattern)
  * Budget enforcement (max_loops, max_workers, max_tokens, max_wall_time)
  * Dynamic worker spawning by manager
  * Validation gates (deterministic + LLM-based)
  * Stall detection and termination

--- A3 Self-Tooling ---
  * Everything in A2, plus:
  * dynamic_tools.enabled = true
  * Agents create tools at runtime (code.execute + tool registration)
  * 

---

## 9. AWP Cheat Sheet

Complete quick reference for everything AWP offers.

In [ ]:
print("""
========================================================================
                        AWP CHEAT SHEET
========================================================================

--- IMPORTS: Parsing ---------------------------------------------------

  from awp.parser import parse_manifest, parse_agent, resolve_templates

--- IMPORTS: Validation ------------------------------------------------

  from awp.validator import (
      validate_schema,     # Validate output_schema.json
      validate_graph,      # Validate DAG structure
      validate_contracts,  # Validate agent output contracts
      validate_rules,      # Run all rules R1-R26
      check_compliance,    # Check autonomy level
      AutonomyLevel,       # Enum: A0-A4
  )

--- IMPORTS: Models ----------------------------------------------------

  from awp.models import (
      # Layer 0: Manifest
      AWPManifest,
      # Layer 1: Identity
      AWPAgent,
      # Layer 2: Capabilities
      ToolsCapability, SkillsCapability, CustomToolsConfig,
      # Layer 3: Communication
      CommunicationConfig, BusConfig, Channel, MessageEnvelope,
      # Layer 4: State & Memory
      StateModel, SharingConfig, PersistenceConfig, MemoryConfig,
      # Layer 5: Orchestration
      AWPOrchestrationConfig, GraphNode, DelegationLoopConfig,
      DelegationBudget, WorkerPolicy, ValidationConfig,
      # Layer 6: Observability
      ObservabilityConfig,
      # Security (cross-cutting)
      SecurityConfig, CircuitBreakerConfig,
      # Common
      SemVer, AgentId, ToolFQN,
  )

--- IMPORTS: Runtime ---------------------------------------------------

  from awp.runtime import (
      WorkflowRunner,         # DAG execution engine (A0-A1)
      DelegationLoopRunner,   # Manager-worker engine (A2-A4)
      StandaloneAgent,        # Base agent class
      ToolRegistry,           # Built-in + custom MCP tools
      CodeExecutor,           # Subprocess Python sandbox
      DockerExecutor,         # Docker container sandbox
      VenvExecutor,           # Virtual environment sandbox
      MessageBus,             # Inter-agent messaging
      SecurityContext,        # Composed security subsystems
      CircuitBreaker,         # Failure threshold + reset
      RateLimiter,            # Sliding window rate limit
      AccessController,       # Per-agent tool ACL
      ObservabilityContext,   # Composed observability
      Tracer,                 # Distributed tracing (JSONL)
      MetricsCollector,       # Counters + histograms
      AuditTrail,             # Hash-chained audit log
      StatePersistence,       # JSON state checkpoints
  )

--- IMPORTS: Data (High-Level API) ------------------------------------

  from awp.data import AgentWorkflow, Source

--- SOURCE FACTORIES (Universal Data Importer) ------------------------

  Source.url("https://example.com/data.csv")              # HTTP fetch
  Source.sql("SELECT ...", dsn="sqlite:///db.sqlite")     # SQL query
  Source.s3("s3://bucket/key.parquet")                    # S3 object
  Source.glob("/data/reports/*.csv", merge="concat")      # File glob
  Source.api("https://api.example.com/v1/data",           # REST API
             method="POST", body={...},
             headers={"Authorization": "Bearer $TOKEN"})
  Source.base64(encoded_str, format="text")               # Base64
  Source.clipboard()                                      # Clipboard

  # $SECRET_NAME in headers/DSNs resolved from secrets={} at runtime

--- CLI COMMANDS -------------------------------------------------------

  awp validate <path>                 # Validate workflow (R1-R26)
  awp compliance <path> --level A2    # Check autonomy level
  awp visualize <path> --format mermaid  # Render DAG
  awp pack <path>                     # Archive as .awp.zip
  awp run <path>                      # Execute workflow

--- AGENT OUTPUT CONTRACT (R17) ----------------------------------------

  Every agent run() must return:
  {
      "agent_name": {
          "confidence": 0.0-1.0,   # REQUIRED by R17
          "field_1": ...,
          "field_2": ...,
      }
  }

--- DELEGATION BUDGET (A2+) --------------------------------------------

  DelegationBudget(
      max_loops=10,            # Max iteration count
      max_total_workers=20,    # Max spawned workers
      max_total_tokens=500000, # Max LLM tokens
      max_wall_time=300,       # Max wall clock seconds
      max_tool_calls=100,      # Max tool invocations
      max_depth=5,             # Max recursive delegation depth
  )

--- COMMON PATTERNS ----------------------------------------------------

  # Parse + validate a workflow
  manifest = parse_manifest("workflow.awp.yaml")
  agents = {a.identity.id: a for yaml_path in ...}
  graph_ok = validate_graph(manifest.orchestration)
  rules_ok = validate_rules(manifest, agents, workflow_path=Path("."))
  compliance = check_compliance(manifest, agents, workflow_path=Path("."),
                                target_level=AutonomyLevel.A2_DELEGATING)

  # Run a data-driven workflow with Source inputs (no YAML needed)
  result = AgentWorkflow(
      inputs={
          "data": df,
          "config": {"threshold": 0.8},
          "remote": Source.url("https://example.com/extra.csv"),
          "db": Source.sql("SELECT * FROM t", dsn="$DB_URL"),
      },
      task="Analyze trends and create a report",
      model="openrouter/anthropic/claude-sonnet-4",
      secrets={"DB_URL": "sqlite:///app.db"},
      max_loops=5,
      code_mode=True,
      tool_creation=True,
  ).run()

  # Wire up runtime components
  registry = ToolRegistry(workflow_dir=Path("."))
  registry.set_code_executor(CodeExecutor(max_timeout=30))
  registry.set_message_bus(MessageBus())
  registry.set_security_context(SecurityContext(...))

========================================================================
""")

---

## 10. Next Steps

Congratulations! You have now seen every major AWP feature in action. Here is where to go next:

### Documentation

| Resource | Path | Description |
|----------|------|-------------|
| Specification | `spec/` | Normative AWP specification (RFC 2119 language) |
| Protocol Docs | `docs/` | Documentation for each of the 7 layers |
| Examples | `examples/` | 12 runnable examples progressing A0 to A4 |
| JSON Schemas | `schemas/` | Machine-readable schema definitions |
| Conformance Tests | `conformance/` | Test fixtures for spec compliance |

### Tutorial Notebooks

| Notebook | Focus |
|----------|-------|
| `01_awp_fundamentals.ipynb` | Models, parsing, validation (no LLM needed) |
| `02_workflow_loading_saving.ipynb` | Load, save, convert, CLI tools |
| `03_runtime_components.ipynb` | Tools, code execution, messaging, security |
| `04_dag_workflow_e2e.ipynb` | End-to-end DAG workflow with real LLM |
| `05_delegation_loop_e2e.ipynb` | End-to-end delegation loop with real LLM |
| **06_full_showcase.ipynb** | This notebook: everything combined |

### Key Examples by Autonomy Level

| Example | Level | Key Feature |
|---------|-------|-------------|
| `01-hello-world` | A0 | Simplest possible workflow |
| `02-research-pipeline` | A1 | Multi-agent DAG with state sharing |
| `05-observable-analytics` | A1 | Full observability stack |
| `06-enterprise` | A1 | Enterprise security features |
| `08-delegation-loop` | A2 | Manager-worker delegation |
| `09-recursive-delegation` | A2+ | Recursive delegation with depth |
| `10-skill-and-tool-generation` | A3 | Dynamic tool creation |
| `12-full-autonomy-test` | A4 | Self-organizing with all features |

### How to Contribute

1. Fork the repository
2. Create a feature branch
3. Run `ruff check .` and `ruff format .` for linting
4. Run `pytest reference/python/tests/` to ensure all tests pass
5. Submit a pull request

### Quick Start for Your Own Workflow

```bash
# Install AWP
pip install -e reference/python/

# Create a new workflow directory
mkdir my-workflow && cd my-workflow

# Copy a template
cp ../examples/workflows/02-research-pipeline/workflow.awp.yaml .

# Validate it
awp validate .

# Check compliance
awp compliance . --level A1

# Run it
awp run .
```

In [27]:
# Final summary
print("=" * 60)
print("         AWP Full Showcase -- Complete!")
print("=" * 60)
print()
print("What we covered:")
print("  [1] Provider setup (Ollama / OpenRouter / Custom)")
print("  [2] Complete AWP feature map (7 layers + autonomy spectrum)")
print("  [3] Built a 3-agent pipeline from scratch with all features")
print("  [4] Validated: schema, graph, rules (R1-R26), compliance")
print("  [5] Ran a data-driven workflow with AgentWorkflow")
print("  [6] Inspected outputs, artifacts, workspace structure")
print("  [7] Wired all runtime components together")
print("  [8] Compared autonomy levels A0-A4 side by side")
print("  [9] Complete AWP cheat sheet")
print("  [10] Next steps and resources")
print()
print("You now know everything AWP has to offer. Go build something!")

         AWP Full Showcase -- Complete!

What we covered:
  [1] Provider setup (Ollama / OpenRouter / Custom)
  [2] Complete AWP feature map (7 layers + autonomy spectrum)
  [3] Built a 3-agent pipeline from scratch with all features
  [4] Validated: schema, graph, rules (R1-R26), compliance
  [5] Ran a data-driven workflow with AgentWorkflow
  [6] Inspected outputs, artifacts, workspace structure
  [7] Wired all runtime components together
  [8] Compared autonomy levels A0-A4 side by side
  [9] Complete AWP cheat sheet
  [10] Next steps and resources

You now know everything AWP has to offer. Go build something!
